# BI Intern Assignment - D2C Analytics

Task 1 (SQL), Task 2 (Python), and the prompt + link for the Task 3 dashboard.
Everything is read straight from the assignment workbook.

## Data load

In [1]:
import pandas as pd
import sqlite3

pd.set_option("display.float_format", "{:,.2f}".format)

FILE = "bi_intern_assignment (2) (2) (1) (1) (1) (2) (3).xlsx"

orders    = pd.read_excel(FILE, sheet_name="📊 orders")
customers = pd.read_excel(FILE, sheet_name="👤 customers")
catalog   = pd.read_excel(FILE, sheet_name="🗂 product_catalog")
pricing   = pd.read_excel(FILE, sheet_name="💰 product_pricing")

orders.head()

,order_id,order_date,customer_id,city,state,category,product_name,quantity,channel,MRP
0,1001,2024-11-23,C103,Chennai,TN,Haircare,Hair Mask,2.00,Website,"1,047.00"
1,1002,2024-10-29,C113,Mumbai,MH,Skincare,Sunscreen SPF50,2.00,Amazon,"1,442.00"
2,1003,2024-04-11,C122,Jaipur,RJ,Makeup,Lipstick,4.00,App,955.00
3,1004,2024-12-23,C113,Chennai,TN,Haircare,Hair Mask,3.00,Website,"1,047.00"
4,1005,2024-07-02,C127,Lucknow,UP,Wellness,Collagen Supplement,4.00,Website,"1,453.00"


In [2]:
# every sheet ends with a blank row and a hint note
orders    = orders[orders["quantity"].notna()].copy()
customers = customers[customers["customer_segment"].notna()].copy()
catalog   = catalog[catalog["mrp"].notna()].copy()
pricing   = pricing[pricing["discount_pct"].notna()].copy()

orders["order_id"]   = orders["order_id"].astype(int)
orders["quantity"]   = orders["quantity"].astype(int)
orders["order_date"] = pd.to_datetime(orders["order_date"])

for name, df in [("orders", orders), ("customers", customers),
                 ("catalog", catalog), ("pricing", pricing)]:
    print(name, df.shape)

orders (50, 10)
customers (26, 6)
catalog (18, 7)
pricing (90, 3)


### Checks on the keys

In [3]:
print("orders with no matching customer:", (~orders["customer_id"].isin(customers["customer_id"])).sum())
print("orders with no matching price row:", len(orders) - len(orders.merge(pricing, on=["product_name", "channel"])))
print("duplicate (product, channel) in pricing:", pricing.duplicated(["product_name", "channel"]).sum())

check = orders.merge(catalog[["product_name", "mrp"]], on="product_name")
print("rows where orders.MRP != catalog.mrp:", (check["MRP"] != check["mrp"]).sum())

orders with no matching customer: 0
orders with no matching price row: 0
duplicate (product, channel) in pricing: 0
rows where orders.MRP != catalog.mrp: 0


---
## Task 1 - SQL

Loaded the sheets into SQLite so the queries actually run and the output can be shown.

In [4]:
con = sqlite3.connect(":memory:")
orders.to_sql("orders", con, index=False)
customers.to_sql("customers", con, index=False)
catalog.to_sql("product_catalog", con, index=False)
pricing.to_sql("product_pricing", con, index=False)

print("tables loaded into sqlite")

tables loaded into sqlite


### QS1

In [5]:
qs1 = """
select c.customer_segment,
       count(*) as total_orders,
       round(avg(o.MRP), 2) as avg_MRP
from orders o
join customers c on o.customer_id = c.customer_id
group by c.customer_segment
order by total_orders desc
"""

pd.read_sql(qs1, con)

,customer_segment,total_orders,avg_MRP
0,Gold,22,"1,327.64"
1,Silver,18,"1,282.11"
2,Bronze,10,"1,282.90"


Gold has the most orders (22 of 50) and the highest average MRP.

### QS2

The same product has a different discount on each channel, so the join needs product_name
and channel. That pair is unique in pricing, so no rows get duplicated.

`revenue = quantity * MRP * (1 - discount_pct / 100)`

In [6]:
qs2 = """
select o.product_name,
       round(sum(o.quantity * o.MRP * (1 - p.discount_pct/100.0)), 2) as total_revenue
from orders o
join product_pricing p
  on o.product_name = p.product_name
 and o.channel = p.channel
group by o.product_name
order by total_revenue desc
"""

pd.read_sql(qs2, con)

,product_name,total_revenue
0,Vitamin C Serum,"30,730.35"
1,Collagen Supplement,"20,429.18"
2,Conditioner,"14,298.75"
3,Ashwagandha,"14,100.00"
4,Night Cream,"13,540.80"
5,Moisturizer,"13,172.60"
6,Hair Mask,"12,145.20"
7,Vitamin D3,"10,751.58"
8,Eyeliner,"10,709.26"
9,Probiotic Blend,"10,605.76"


### QS3

Taking only the product/channel combinations that were actually ordered.

In [7]:
qs3 = """
select o.channel,
       count(*) as total_orders,
       round(avg(p.discount_pct), 2) as avg_discount_pct,
       round(sum(o.quantity * o.MRP * p.discount_pct/100.0), 2) as discount_given
from orders o
join product_pricing p
  on o.product_name = p.product_name
 and o.channel = p.channel
group by o.channel
order by avg_discount_pct desc
"""

pd.read_sql(qs3, con)

,channel,total_orders,avg_discount_pct,discount_given
0,Amazon,7,17.43,"2,734.04"
1,Instagram,11,11.36,"5,288.45"
2,Website,8,9.25,"2,854.76"
3,WhatsApp,9,7.11,"1,645.84"
4,App,15,5.53,"4,764.05"


**Amazon**, at 17.43% average discount across 7 orders. App is the lowest at 5.53%.
Same answer if you weight by order value (Amazon gives away 16.3% of its list value).

---
## Task 2 - Python

QP1 is the only task on the Python sheet (the cover tab also lists it as 1 Python task).

Category | Total Revenue | Orders | Avg Order Value. Same revenue logic as QS2.

In [8]:
order_revenue = orders.merge(pricing, on=["product_name", "channel"], how="left")
print("rows after join:", len(order_revenue))   # should still be 50

order_revenue["revenue"] = (order_revenue["quantity"] * order_revenue["MRP"]
                            * (1 - order_revenue["discount_pct"] / 100))

order_revenue[["order_id", "category", "product_name", "channel",
               "quantity", "MRP", "discount_pct", "revenue"]].head()

rows after join: 50


,order_id,category,product_name,channel,quantity,MRP,discount_pct,revenue
0,1001,Haircare,Hair Mask,Website,2,"1,047.00",12.00,"1,842.72"
1,1002,Skincare,Sunscreen SPF50,Amazon,2,"1,442.00",20.00,"2,307.20"
2,1003,Makeup,Lipstick,App,4,955.00,5.00,"3,629.00"
3,1004,Haircare,Hair Mask,Website,3,"1,047.00",12.00,"2,764.08"
4,1005,Wellness,Collagen Supplement,Website,4,"1,453.00",12.00,"5,114.56"


In [9]:
# Orders is the number of orders, not units sold
category_summary = (order_revenue
                    .groupby("category")
                    .agg(**{"Total Revenue": ("revenue", "sum"),
                            "Orders": ("order_id", "nunique")})
                    .reset_index()
                    .rename(columns={"category": "Category"}))

category_summary["Avg Order Value"] = (category_summary["Total Revenue"]
                                       / category_summary["Orders"])

category_summary = (category_summary[["Category", "Total Revenue", "Orders", "Avg Order Value"]]
                    .sort_values("Total Revenue", ascending=False)
                    .round(2)
                    .reset_index(drop=True))

category_summary

,Category,Total Revenue,Orders,Avg Order Value
0,Wellness,"65,337.06",16,"4,083.57"
1,Skincare,"64,722.59",15,"4,314.84"
2,Haircare,"38,013.82",14,"2,715.27"
3,Makeup,"15,283.39",5,"3,056.68"


In [10]:
print("Total revenue:", round(category_summary["Total Revenue"].sum(), 2))
print("Total orders:", category_summary["Orders"].sum())

Total revenue: 183356.86
Total orders: 50


Wellness and Skincare are nearly level on revenue but get there differently - Wellness on
order count, Skincare on a higher average order value. Makeup is only 5 orders.

---
## Task 3 - Dashboard

Dashboard file: `dashboard.html`

It is a single HTML file that runs in any browser with no server. The 50 orders are embedded
in the file and every KPI is calculated in JavaScript, so the numbers match this notebook.

### Initial prompt given to the LLM

> Build me a single-page HTML dashboard for a D2C beauty and wellness brand covering channel
> and product performance. It has to be one standalone .html file that opens straight in a
> browser - no server, no npm, no backend, and no CDN either, so please write the charts as
> plain SVG/CSS instead of pulling in a chart library.
>
> Use this data (50 orders, embed it as a JSON array inside the file): order_id, order_date,
> customer_id, customer_segment, city, state, category, product_name, quantity, channel, MRP,
> discount_pct. Revenue for an order is quantity * MRP * (1 - discount_pct/100). Calculate
> everything in JavaScript from the raw rows, don't hardcode any totals.
>
> Layout I want:
> - KPI cards at the top: total revenue, orders, average order value, units sold, average
>   discount % and the discount value given away
> - Channel performance: revenue by channel, plus a table with orders, units, revenue, AOV
>   and average discount % per channel
> - Product performance: top products by revenue, and the revenue split by category
> - Monthly revenue trend for 2024
> - A short "what this tells us" section with 3-4 observations taken from the numbers
>
> Keep it clean and professional, something I can send to my manager - neutral colours,
> readable tables, sensible spacing, works on a laptop and on a phone. Don't overload it
> with charts.